In [1]:
import os
from typing import TypedDict,  List, Literal
from typing_extensions import Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import os

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1", # Added /v1 for standard OpenAI compatibility
    api_key=os.getenv("OPENROUTER_API"),                    # Placeholder to prevent auth errors
    temperature=0,
   )




In [12]:
@tool
def search_web(query:str)->str:
    """Search the web for information."""

    search = TavilySearchResults(max_results=3)
    results=search.invoke(query)
    return str(results)

In [13]:
def add(a:float,b:float)->float:
    "Add two numbers"
    return a+b

def multiply(a:float,b:float)->float:
    "multiply two numbers"
    return a * b    

In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import json

In [24]:
mcp_client = MultiServerMCPClient(
        json.load(open("new_tools.json"))
    )

li=["math","research"]
mcptools = []

for name in li:
    tools = mcp_client.get_tools(server_name=name)
    mcptools.extend(tools)



C:\Users\DNFH3173\AppData\Local\Temp\ipykernel_6728\4241342303.py:9: RuntimeWarning: coroutine 'MultiServerMCPClient.get_tools' was never awaited
  tools = mcp_client.get_tools(server_name=name)


TypeError: 'coroutine' object is not iterable

In [ ]:
import asyncio

tool_lists = await asyncio.gather(
    *[mcp_client.get_tools(server_name=name) for name in li]
)

# Flatten list
mcptools = [tool for sublist in tool_lists for tool in sublist]

UnsupportedOperation: fileno

In [ ]:
math_agent=create_react_agent(
    model=llm,
    tools=[add,multiply],
    name="Math_Expert",
    prompt="You are a math expert, Always use one tool at a time. "
)

research_agent=create_react_agent(
    model=llm,
    tools=mcptools,
    name="Research_Agent",
    prompt="You are a world class researcher with access to web search. Do not do any math."
)

In [45]:
from langgraph_supervisor import create_supervisor

workflow = create_supervisor(
    [research_agent,math_agent],
    model=llm,
    prompt=(
        "You are a team supervisor managing a research expert and a math expert."
        "For current events, use research_agent"
        "For math problems, use math_agent."
    )
)

In [46]:
app= workflow.compile()
result=app.invoke({
    "messages":[
        {
            "role":"user",
            "content":"do some latest research on AI"
        }
    ]
})

In [47]:
result

{'messages': [HumanMessage(content='do some latest research on AI', additional_kwargs={}, response_metadata={}, id='4bbaf762-e359-44ae-9c91-9f658e8e6787'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 183, 'total_tokens': 224, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 14, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.045247378, 'prompt_time': 0.007894812, 'completion_time': 0.086922685, 'total_time': 0.094817497}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'id': 'chatcmpl-471f0405-a554-4232-b2fd-71554e163ddf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019cb3ab-69ee-7e50-bcc4-57603fa7f892-0', tool_calls=[{'name': 'transfer_to_research_agent', 'args': {'': {}}, 'id': 

In [50]:
print(result["messages"][-1].content)

**Latest AI Research Highlights (2023‑2024)**  

Below is a concise, citation‑rich overview of the most notable research themes that have emerged in the last 12‑18 months.  The material is drawn from peer‑reviewed articles, pre‑prints, and high‑impact industry reports that were publicly available as of early 2024.

---

## 1. Foundation & Multimodal Models – The “Generalist” Turn  

| Sub‑area | Key Findings (2023‑24) | Representative Papers / Reports |
|----------|-----------------------|---------------------------------|
| **Vision‑Language Foundations** | • Large‑scale vision‑language models (VLMs) now achieve *human‑level* performance on a range of zero‑shot tasks (image captioning, visual reasoning, medical image analysis). <br>• Self‑supervised pre‑training on billions of image‑text pairs dramatically reduces the need for task‑specific labels. | • **DINOv2** – Oquab *et al.*, *Trans. Mach. Learn. Res.* (2024) – shows that purely self‑supervised visual transformers learn robust, t